<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%BB%BA%E7%AB%8B%E5%9F%BA%E6%A8%A1%E5%9E%8B/AI_model/AIModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
def load_data() -> pd.DataFrame:
  uploaded = files.upload()

  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
  file_name = next(iter(uploaded))
  df = pd.read_csv(file_name)
  return df

In [ ]:
df1 = load_data()
df1.info()

Saving phishing_dataset_Gemini_text_success.csv to phishing_dataset_Gemini_text_success (1).csv
User uploaded file "phishing_dataset_Gemini_text_success (1).csv" with length 40333527 bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11430 entries, 0 to 11429
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   url                                11430 non-null  object 
 1   target                             11430 non-null  int64  
 2   visible_text                       11059 non-null  object 
 3   ai_status                          6211 non-null   object 
 4   fetch_status                       11430 non-null  object 
 5   creates_urgency                    6156 non-null   object 
 6   uses_threats                       6156 non-null   object 
 7   requests_sensitive_info            6156 non-null   object 
 8   offers_unrealistic_rewards         6156 non-null   object

In [ ]:
df2 = load_data()
df2.info()

Saving phishing_dataset_expansion_1_Gemini_text_success.csv to phishing_dataset_expansion_1_Gemini_text_success (1).csv
User uploaded file "phishing_dataset_expansion_1_Gemini_text_success (1).csv" with length 200995366 bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   url                                50000 non-null  object 
 1   target                             50000 non-null  int64  
 2   visible_text                       50000 non-null  object 
 3   ai_status                          27008 non-null  object 
 4   fetch_status                       50000 non-null  object 
 5   creates_urgency                    26706 non-null  object 
 6   uses_threats                       26706 non-null  object 
 7   requests_sensitive_info            26706 non-null  object 
 8   offers_unrealistic_r

In [ ]:
df1 = df1[df1['ai_status'] == 'AI_SUCCESS']
df1.drop(['url'], axis=1, inplace=True) #, 'visible_text'
df2 = df2[df2['ai_status'] == 'AI_SUCCESS']
df2.drop(['url', 'visible_text'], axis=1, inplace=True)
df1.drop(['ai_status', 'fetch_status'], axis=1, inplace=True)
df2.drop(['ai_status', 'fetch_status'], axis=1, inplace=True)
df1.info()
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6156 entries, 1 to 11055
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   target                             6156 non-null   int64  
 1   visible_text                       6156 non-null   object 
 2   creates_urgency                    6156 non-null   object 
 3   uses_threats                       6156 non-null   object 
 4   requests_sensitive_info            6156 non-null   object 
 5   offers_unrealistic_rewards         6156 non-null   object 
 6   has_spelling_grammar_errors        6156 non-null   object 
 7   impersonated_brand                 1555 non-null   object 
 8   has_valid_copyright_year           6156 non-null   object 
 9   is_content_login_focused           6156 non-null   object 
 10  has_rich_navigation                6156 non-null   object 
 11  has_physical_address               6156 non-null   object 
 

In [ ]:
bool_col = ['creates_urgency', 'uses_threats', 'requests_sensitive_info',
            'offers_unrealistic_rewards', 'has_spelling_grammar_errors',
            'has_valid_copyright_year', 'is_content_login_focused',
            'has_rich_navigation', 'has_physical_address', 'has_phone_number']

df1[bool_col] = df1[bool_col].astype(int)
df2[bool_col] = df2[bool_col].astype(int)
df1['impersonated_brand'] = [1 if k else 0 for k in df1['impersonated_brand'].notnull()]
df2['impersonated_brand'] = [1 if k else 0 for k in df2['impersonated_brand'].notnull()]

In [ ]:
df1.info()
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6156 entries, 1 to 11055
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   target                             6156 non-null   int64  
 1   creates_urgency                    6156 non-null   int64  
 2   uses_threats                       6156 non-null   int64  
 3   requests_sensitive_info            6156 non-null   int64  
 4   offers_unrealistic_rewards         6156 non-null   int64  
 5   has_spelling_grammar_errors        6156 non-null   int64  
 6   impersonated_brand                 6156 non-null   int64  
 7   has_valid_copyright_year           6156 non-null   int64  
 8   is_content_login_focused           6156 non-null   int64  
 9   has_rich_navigation                6156 non-null   int64  
 10  has_physical_address               6156 non-null   int64  
 11  has_phone_number                   6156 non-null   int64  
 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
X = df1.drop('target', axis=1)
y = df1['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 建立和訓練 XGBoost 模型 ---
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
model = XGBClassifier(
    scale_pos_weight=ratio,
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False
)

print("開始訓練 XGBoost 模型...")
model.fit(X_train, y_train)
print("模型訓練完成。")

# --- 預測和評估模型 ---
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # 獲取預測為正類 (1) 的機率

# 計算評估指標
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n模型評估結果 (測試集):")
print(f"準確度 (Accuracy): {accuracy:.4f}")
print(f"精確度 (Precision): {precision:.4f}")
print(f"召回率 (Recall): {recall:.4f}")
print(f"F1 分數 (F1 Score): {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

# --- 查看特徵重要性 (XGBoost 內建) ---
feature_importances = model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

pd.set_option('display.max_rows', None)
print("\n特徵重要性:")
print(importance_df)



開始訓練 XGBoost 模型...
模型訓練完成。

模型評估結果 (測試集):
準確度 (Accuracy): 0.7427
精確度 (Precision): 0.5218
召回率 (Recall): 0.7426
F1 分數 (F1 Score): 0.6129
ROC AUC: 0.8038

特徵重要性:
                              Feature  Importance
8                 has_rich_navigation    0.233240
11          content_consistency_score    0.170662
13  overall_phishing_likelihood_score    0.129000
5                  impersonated_brand    0.087081
6            has_valid_copyright_year    0.084917
10                   has_phone_number    0.084441
9                has_physical_address    0.052635
12     language_professionalism_score    0.037304
3          offers_unrealistic_rewards    0.035403
7            is_content_login_focused    0.034993
2             requests_sensitive_info    0.026597
1                        uses_threats    0.019462
0                     creates_urgency    0.004267
4         has_spelling_grammar_errors    0.000000


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:48:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
X = df2.drop('target', axis=1)
y = df2['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 建立和訓練 XGBoost 模型 ---
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
model = XGBClassifier(
    scale_pos_weight=ratio,
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False
)

print("開始訓練 XGBoost 模型...")
model.fit(X_train, y_train)
print("模型訓練完成。")

# --- 預測和評估模型 ---
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # 獲取預測為正類 (1) 的機率

# 計算評估指標
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n模型評估結果 (測試集):")
print(f"準確度 (Accuracy): {accuracy:.4f}")
print(f"精確度 (Precision): {precision:.4f}")
print(f"召回率 (Recall): {recall:.4f}")
print(f"F1 分數 (F1 Score): {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

# --- 查看特徵重要性 (XGBoost 內建) ---
feature_importances = model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

pd.set_option('display.max_rows', None)
print("\n特徵重要性:")
print(importance_df)



開始訓練 XGBoost 模型...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:48:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


模型訓練完成。

模型評估結果 (測試集):
準確度 (Accuracy): 0.8209
精確度 (Precision): 0.9409
召回率 (Recall): 0.8499
F1 分數 (F1 Score): 0.8931
ROC AUC: 0.8045

特徵重要性:
                              Feature  Importance
8                 has_rich_navigation    0.825289
13  overall_phishing_likelihood_score    0.030364
10                   has_phone_number    0.024636
11          content_consistency_score    0.024474
9                has_physical_address    0.017321
6            has_valid_copyright_year    0.015705
3          offers_unrealistic_rewards    0.013588
5                  impersonated_brand    0.012106
12     language_professionalism_score    0.009074
2             requests_sensitive_info    0.008364
1                        uses_threats    0.006295
0                     creates_urgency    0.005155
7            is_content_login_focused    0.004518
4         has_spelling_grammar_errors    0.003109


In [ ]:
df3 = pd.concat([df1, df2], axis=0)

In [ ]:
df3['text_length'] = df3['visible_text'].apply(lambda x: len(str(x)))

In [ ]:
df3.drop('visible_text', axis=1, inplace=True)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
X = df3.drop('target', axis=1)
y = df3['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 建立和訓練 XGBoost 模型 ---
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
model = XGBClassifier(
    scale_pos_weight=ratio,
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False
)
# model = XGBClassifier(
#     scale_pos_weight=ratio,
#     objective='binary:logistic',
#     eval_metric='auc',
#     n_estimators=1200,
#     learning_rate=0.05,
#     min_child_weight=3,   # 避免樹切分到太細微的雜訊
#     subsample=0.8,        # 每次長樹只隨機抽取 80% 的資料 (增加模型泛化能力)
#     colsample_bytree=0.8, # 每次長樹只隨機抽取 80% 的特徵
#     max_depth=8,
#     random_state=42,
#     use_label_encoder=False
# )

print("開始訓練 XGBoost 模型...")
model.fit(X_train, y_train)
print("模型訓練完成。")

# --- 預測和評估模型 ---
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # 獲取預測為正類 (1) 的機率

# 計算評估指標
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n模型評估結果 (測試集):")
print(f"準確度 (Accuracy): {accuracy:.4f}")
print(f"精確度 (Precision): {precision:.4f}")
print(f"召回率 (Recall): {recall:.4f}")
print(f"F1 分數 (F1 Score): {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

# --- 查看特徵重要性 (XGBoost 內建) ---
feature_importances = model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

pd.set_option('display.max_rows', None)
print("\n特徵重要性:")
print(importance_df)



開始訓練 XGBoost 模型...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:41:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


模型訓練完成。

模型評估結果 (測試集):
準確度 (Accuracy): 0.8442
精確度 (Precision): 0.9320
召回率 (Recall): 0.8596
F1 分數 (F1 Score): 0.8943
ROC AUC: 0.8921

特徵重要性:
                              Feature  Importance
8                 has_rich_navigation    0.671563
14                        text_length    0.206722
13  overall_phishing_likelihood_score    0.023630
11          content_consistency_score    0.022447
10                   has_phone_number    0.013645
9                has_physical_address    0.010023
6            has_valid_copyright_year    0.009974
3          offers_unrealistic_rewards    0.007098
2             requests_sensitive_info    0.006857
12     language_professionalism_score    0.006642
1                        uses_threats    0.006622
5                  impersonated_brand    0.005467
7            is_content_login_focused    0.004341
0                     creates_urgency    0.002624
4         has_spelling_grammar_errors    0.002344


In [ ]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32862 entries, 1 to 49996
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   target                             32862 non-null  int64  
 1   creates_urgency                    32862 non-null  int64  
 2   uses_threats                       32862 non-null  int64  
 3   requests_sensitive_info            32862 non-null  int64  
 4   offers_unrealistic_rewards         32862 non-null  int64  
 5   has_spelling_grammar_errors        32862 non-null  int64  
 6   impersonated_brand                 32862 non-null  int64  
 7   has_valid_copyright_year           32862 non-null  int64  
 8   is_content_login_focused           32862 non-null  int64  
 9   has_rich_navigation                32862 non-null  int64  
 10  has_physical_address               32862 non-null  int64  
 11  has_phone_number                   32862 non-null  int64  


In [ ]:
# 假設您的標籤欄位叫做 'target' (1: 釣魚, 0: 正常)
print("--- 資料集 1 ---")
print(df1.groupby('has_rich_navigation')['target'].mean())

print("\n--- 資料集 2 ---")
print(df2.groupby('has_rich_navigation')['target'].mean())

--- 資料集 1 ---
has_rich_navigation
0    0.413440
1    0.170358
Name: target, dtype: float64

--- 資料集 2 ---
has_rich_navigation
0    0.587650
1    0.936958
Name: target, dtype: float64


In [ ]:
model.save_model("ai_xgb_v1.json")